# OMOP Measurement Table

Transforms FHIR Observation resources (laboratory results, vital signs) into OMOP CDM `measurement` table.

## Mapping: FHIR Observation → OMOP Measurement

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| measurement_id | Observation.id | Hash to integer |
| person_id | Observation.subject | Reference to person |
| measurement_concept_id | Observation.code | Map LOINC to OMOP |
| measurement_date | Observation.effectiveDateTime | Extract date |
| measurement_datetime | Observation.effectiveDateTime | Full timestamp |
| measurement_type_concept_id | - | 32856 (Lab result) |
| value_as_number | Observation.valueQuantity.value | Numeric value |
| value_as_concept_id | Observation.valueCodeableConcept | Coded value |
| unit_concept_id | Observation.valueQuantity.unit | UCUM to OMOP |
| range_low | Observation.referenceRange.low | Lower bound |
| range_high | Observation.referenceRange.high | Upper bound |
| measurement_source_value | Observation.code.coding[0].code | Original code |

## Categories Mapped to Measurement

| FHIR Category | Description |
|---------------|-------------|
| laboratory | Lab test results |
| vital-signs | Vital signs (BP, HR, Temp, etc.) |

## Vocabulary: LOINC → OMOP

LOINC is the standard vocabulary for measurements in OMOP CDM.

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## Create Measurement Streaming Table

In [ ]:
DECLARE OR REPLACE VARIABLE create_measurement_stmt STRING;

SET VARIABLE create_measurement_stmt = "
CREATE OR REFRESH STREAMING TABLE measurement (
  -- Primary key
  measurement_id BIGINT NOT NULL COMMENT 'Unique measurement identifier'
  
  -- Person reference
  ,person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  
  -- Measurement coding
  ,measurement_concept_id INT NOT NULL COMMENT 'OMOP standard concept for measurement (LOINC)'
  
  -- Dates
  ,measurement_date DATE NOT NULL COMMENT 'Measurement date'
  ,measurement_datetime TIMESTAMP COMMENT 'Measurement datetime'
  ,measurement_time STRING COMMENT 'Time of measurement'
  
  -- Type
  ,measurement_type_concept_id INT NOT NULL DEFAULT 32856 COMMENT 'Type: 32856=Lab, 32817=EHR'
  
  -- Operator
  ,operator_concept_id INT DEFAULT 0 COMMENT 'Operator (=, <, >, etc.)'
  
  -- Values
  ,value_as_number DECIMAL(18,6) COMMENT 'Numeric result value'
  ,value_as_concept_id INT DEFAULT 0 COMMENT 'Coded result value'
  
  -- Units
  ,unit_concept_id INT DEFAULT 0 COMMENT 'Unit concept (UCUM)'
  ,unit_source_value STRING COMMENT 'Original unit value'
  
  -- Reference range
  ,range_low DECIMAL(18,6) COMMENT 'Lower bound of normal range'
  ,range_high DECIMAL(18,6) COMMENT 'Upper bound of normal range'
  
  -- References
  ,provider_id BIGINT COMMENT 'Reference to provider table'
  ,visit_occurrence_id BIGINT COMMENT 'Reference to visit_occurrence table'
  ,visit_detail_id BIGINT COMMENT 'Reference to visit_detail table'
  
  -- Source values
  ,measurement_source_value STRING COMMENT 'Original measurement code'
  ,measurement_source_concept_id INT DEFAULT 0 COMMENT 'Source vocabulary concept'
  ,value_source_value STRING COMMENT 'Original result value'
  
  -- Additional info
  ,measurement_code_system STRING COMMENT 'Source code system (LOINC, etc.)'
  ,measurement_display STRING COMMENT 'Display text for measurement'
  ,category STRING COMMENT 'FHIR category (laboratory, vital-signs)'
  
  -- Lineage
  ,fhir_observation_uuid STRING COMMENT 'Original FHIR Observation UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Measurement table - Labs and vitals from FHIR Observation resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer measurement_id
  ABS(HASH(COALESCE(id::STRING, observation_uuid))) AS measurement_id
  
  -- Person reference
  ,ABS(HASH(
    COALESCE(
      REGEXP_EXTRACT(subject:reference::STRING, 'Patient/(.+)', 1),
      subject:reference::STRING
    )
  )) AS person_id
  
  -- Measurement concept - placeholder using hash of LOINC code
  -- In production, join to OMOP vocabulary tables for proper concept_id
  ,COALESCE(
    ABS(HASH(code:coding[0]:code::STRING)) % 2000000000,
    0
  ) AS measurement_concept_id
  
  -- Dates
  ,COALESCE(
    CAST(TRY_CAST(effectiveDateTime::STRING AS TIMESTAMP) AS DATE),
    CAST(TRY_CAST(effectivePeriod:start::STRING AS TIMESTAMP) AS DATE),
    CAST(TRY_CAST(issued::STRING AS TIMESTAMP) AS DATE),
    CURRENT_DATE()
  ) AS measurement_date
  ,COALESCE(
    TRY_CAST(effectiveDateTime::STRING AS TIMESTAMP),
    TRY_CAST(effectivePeriod:start::STRING AS TIMESTAMP),
    TRY_CAST(issued::STRING AS TIMESTAMP)
  ) AS measurement_datetime
  ,NULL AS measurement_time
  
  -- Type (Lab=32856, Vital Signs can use 32817 for EHR)
  ,CASE 
    WHEN category[0]:coding[0]:code::STRING = 'laboratory' THEN 32856
    WHEN category[0]:coding[0]:code::STRING = 'vital-signs' THEN 32817
    ELSE 32856
  END AS measurement_type_concept_id
  
  -- Operator
  ,CASE valueQuantity:comparator::STRING
    WHEN '<' THEN 4171756
    WHEN '<=' THEN 4171754
    WHEN '>=' THEN 4171755
    WHEN '>' THEN 4172704
    ELSE 4172703  -- '=' (equals)
  END AS operator_concept_id
  
  -- Values
  ,TRY_CAST(valueQuantity:value::STRING AS DECIMAL(18,6)) AS value_as_number
  ,CASE 
    WHEN valueCodeableConcept IS NOT NULL THEN
      ABS(HASH(valueCodeableConcept:coding[0]:code::STRING)) % 2000000000
    ELSE 0
  END AS value_as_concept_id
  
  -- Units
  ,0 AS unit_concept_id  -- Would need UCUM to OMOP mapping
  ,COALESCE(valueQuantity:unit::STRING, valueQuantity:code::STRING) AS unit_source_value
  
  -- Reference range
  ,TRY_CAST(referenceRange[0]:low:value::STRING AS DECIMAL(18,6)) AS range_low
  ,TRY_CAST(referenceRange[0]:high:value::STRING AS DECIMAL(18,6)) AS range_high
  
  -- References
  ,CASE 
    WHEN performer[0]:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(performer[0]:reference::STRING, 'Practitioner/(.+)', 1)))
    ELSE NULL
  END AS provider_id
  ,CASE 
    WHEN encounter:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(encounter:reference::STRING, 'Encounter/(.+)', 1)))
    ELSE NULL
  END AS visit_occurrence_id
  ,NULL AS visit_detail_id
  
  -- Source values
  ,code:coding[0]:code::STRING AS measurement_source_value
  ,0 AS measurement_source_concept_id
  ,COALESCE(
    valueQuantity:value::STRING,
    valueCodeableConcept:coding[0]:display::STRING,
    valueString::STRING
  ) AS value_source_value
  
  -- Additional info
  ,code:coding[0]:system::STRING AS measurement_code_system
  ,COALESCE(code:coding[0]:display::STRING, code:text::STRING) AS measurement_display
  ,category[0]:coding[0]:code::STRING AS category
  
  -- Lineage
  ,observation_uuid AS fhir_observation_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".observation)
WHERE status::STRING IN ('final', 'amended', 'corrected', 'preliminary')
  AND (
    category[0]:coding[0]:code::STRING IN ('laboratory', 'vital-signs')
    OR code:coding[0]:system::STRING LIKE '%loinc%'
  )
";

SELECT create_measurement_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_measurement_stmt;

In [ ]:
-- Verify measurement table
SELECT 
  measurement_id,
  person_id,
  measurement_concept_id,
  measurement_date,
  value_as_number,
  unit_source_value,
  measurement_source_value,
  measurement_display,
  category
FROM measurement
LIMIT 10;

In [ ]:
-- Measurements by category
SELECT 
  category,
  COUNT(*) AS count
FROM measurement
GROUP BY category
ORDER BY count DESC;

In [ ]:
-- Top measurements (LOINC codes)
SELECT 
  measurement_source_value,
  measurement_display,
  COUNT(*) AS occurrences,
  AVG(value_as_number) AS avg_value,
  MIN(value_as_number) AS min_value,
  MAX(value_as_number) AS max_value
FROM measurement
WHERE value_as_number IS NOT NULL
GROUP BY measurement_source_value, measurement_display
ORDER BY occurrences DESC
LIMIT 20;